In [1]:
!pip install fairtfm

  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 3.9 MB/s eta 0:00:00
Using cached click-8.5.0-py3-none-any.whl (125 kB)
Using cached hf_xet-1.6.0-cp38-abi3-macosx_11_0_arm64.whl (3.9 MB)
Using cached fsspec-2026.7.0-py3-none-any.whl (206 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.2/127.2 MB 9.4 MB/s eta 0:00:0000:0100:01
Using cached networkx-3.4.2-py3-none-any.whl (1.7 MB)
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)
Using cached sympy-1.1

In [1]:
from fairtfm import FairTFMClassifier

In [1]:
from fairlearn.datasets import fetch_adult
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.linear_model import LogisticRegression
import pandas as pd

In [11]:
X_raw, y = fetch_adult(as_frame=True, return_X_y=True)

In [12]:
X_raw.head(5)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States


In [13]:
s = X_raw['sex']
X = X_raw.drop(columns=['sex'])
X = pd.get_dummies(X, drop_first=True)
 
y = LabelEncoder().fit_transform(y)
s = LabelEncoder().fit_transform(s)





X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(X, y, s, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [24]:
# subsample for faster inference
train_sample_size, test_sample_size = 1000, 500 
y_train = y_train[:train_sample_size] 
s_train = s_train[:train_sample_size]
X_train = X_train[:train_sample_size]
X_test = X_test[:test_sample_size]
y_test = y_test[:test_sample_size]
s_test = s_test[:test_sample_size]

In [25]:
lr = LogisticRegression()

lr.fit(X_train, y_train)
y_prob = lr.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

In [26]:
from fairlearn.metrics import demographic_parity_difference 
from sklearn.metrics import accuracy_score, roc_auc_score

In [27]:
acc= accuracy_score(y_test, y_pred)
dp_diff = demographic_parity_difference(y_test, y_pred, sensitive_features=s_test)
auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy: {acc:.4f}")
print(f"Demographic Parity Difference: {dp_diff:.4f}")
print(f"AUC: {auc:.4f}")

Accuracy: 0.8240
Demographic Parity Difference: 0.0823
AUC: 0.8661
